<a href="https://colab.research.google.com/github/tony0990/Data_mining/blob/main/FINAL_DASHBOARD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# ============================================================
# ONE COLAB CELL - PROFESSIONAL DATA MINING DASHBOARD
# Version: Dynamic Compare Metrics
# ------------------------------------------------------------
# What this cell does:
# 1) Uses the uploaded association/pattern CSV files directly.
# 2) Uses repository_link_analysis_results.csv for PageRank and Degree Centrality.
# 3) Keeps PageRank and Degree Centrality separated, not combined.
# 4) Maps repo_id values to real repository names only for display.
# 5) Builds a clean local HTML dashboard, not Streamlit and not a web server.
# 6) Removes emojis from the dashboard.
# ============================================================

import warnings
warnings.filterwarnings("ignore")

!pip -q install plotly pandas numpy networkx

import os
import re
import json
import math
import itertools
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from google.colab import files

# ============================================================
# CONFIG
# ============================================================

OUTPUT_HTML = "professional_data_mining_dashboard_DYNAMIC_COMPARE_METRICS.html"
MAX_GRAPH_REPOS = 70
MIN_REPO_JACCARD_EDGE = 0.28
TOP_ROWS = 50

pio.templates.default = "plotly_dark"

COLOR_BG = "#0B1120"
COLOR_PANEL = "#111827"
COLOR_TEXT = "#E5E7EB"
COLOR_MUTED = "#94A3B8"
COLOR_CYAN = "#38BDF8"
COLOR_PURPLE = "#8B5CF6"
COLOR_GREEN = "#22C55E"
COLOR_AMBER = "#F59E0B"
COLOR_RED = "#EF4444"

# ============================================================
# FILE LOADING HELPERS
# ============================================================

def find_file(possible_names):
    local_files = os.listdir(".")
    lower_map = {f.lower(): f for f in local_files}

    for name in possible_names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    for f in local_files:
        f_low = f.lower()
        for name in possible_names:
            base = name.lower().replace(".csv", "")
            if f_low.startswith(base) and f_low.endswith(".csv"):
                return f

    return None


def safe_read_csv(possible_names, required=False):
    path = find_file(possible_names)
    if path is None:
        if required:
            raise FileNotFoundError(f"Required CSV not found. Tried: {possible_names}")
        return pd.DataFrame(), None

    df = pd.read_csv(path)
    return df, path

# ============================================================
# LOAD FILES
# ============================================================

ranking_df, ranking_path = safe_read_csv([
    "top_repositories_ranking.csv",
    "top_repositories_ranking(1).csv",
    "top_repositories_ranking(2).csv"
], required=True)

link_df, link_path = safe_read_csv([
    "repository_link_analysis_results.csv",
    "repository_link_analysis_results(1).csv",
    "repository_link_analysis_results(2).csv"
], required=True)

rules_df, rules_path = safe_read_csv([
    "association_rules.csv",
    "association_rules(1).csv",
    "association_rules(2).csv"
], required=True)

strong_rules_df, strong_rules_path = safe_read_csv([
    "strong_association_rules.csv",
    "strong_association_rules(1).csv",
    "strong_association_rules(2).csv"
], required=False)

apriori_df, apriori_path = safe_read_csv([
    "apriori_frequent_itemsets.csv",
    "apriori_frequent_itemsets(1).csv",
    "apriori_frequent_itemsets(2).csv"
], required=False)

fpgrowth_df, fpgrowth_path = safe_read_csv([
    "fpgrowth_frequent_itemsets.csv",
    "fpgrowth_frequent_itemsets(1).csv",
    "fpgrowth_frequent_itemsets(2).csv"
], required=False)

freq_combo_df, freq_combo_path = safe_read_csv([
    "frequent_technology_combinations.csv",
    "frequent_technology_combinations(1).csv",
    "frequent_technology_combinations(2).csv"
], required=False)

speed_df, speed_path = safe_read_csv([
    "apriori_vs_fpgrowth_speed_comparison.csv",
    "apriori_vs_fpgrowth_speed_comparison(1).csv",
    "apriori_vs_fpgrowth_speed_comparison(2).csv"
], required=False)

pair_df, pair_path = safe_read_csv([
    "technology_pair_cooccurrence.csv",
    "technology_pair_cooccurrence(1).csv",
    "technology_pair_cooccurrence(2).csv"
], required=False)

print("Loaded files:")
for label, path in [
    ("Ranking", ranking_path),
    ("Link Analysis", link_path),
    ("Association Rules", rules_path),
    ("Strong Rules", strong_rules_path),
    ("Apriori", apriori_path),
    ("FP-Growth", fpgrowth_path),
    ("Frequent Combinations", freq_combo_path),
    ("Speed", speed_path),
    ("Pair Co-occurrence", pair_path),
]:
    print(f"- {label}: {path}")

# ============================================================
# VALIDATE REQUIRED COLUMNS
# ============================================================

required_link_cols = {"repository", "pagerank_score", "degree_centrality"}
missing_link = required_link_cols - set(link_df.columns)
if missing_link:
    raise ValueError(f"repository_link_analysis_results.csv is missing columns: {missing_link}")

if "repo_id" not in ranking_df.columns:
    raise ValueError("top_repositories_ranking.csv must contain repo_id column.")

if "technologies" not in ranking_df.columns:
    raise ValueError("top_repositories_ranking.csv must contain technologies column.")

# ============================================================
# MAP repo_id TO REAL REPOSITORY NAMES ONLY
# ============================================================

ranking_df = ranking_df.reset_index(drop=True).copy()
link_df = link_df.reset_index(drop=True).copy()


def extract_repo_number(repo_id):
    text = str(repo_id).strip()
    match = re.search(r"repo_(\d+)", text)
    if match:
        return int(match.group(1))
    match = re.search(r"(\d+)", text)
    if match:
        return int(match.group(1))
    return None


def map_repo_id_to_name(repo_id):
    idx = extract_repo_number(repo_id)

    # Main mapping: repo_172 means row index 172 in link_df.
    if idx is not None and 0 <= idx < len(link_df):
        name = link_df.iloc[idx]["repository"]
        if pd.notna(name) and str(name).strip():
            return str(name).strip()

    # Fallback for 1-based ids such as repo_3562 when len = 3562.
    if idx is not None and 1 <= idx <= len(link_df):
        name = link_df.iloc[idx - 1]["repository"]
        if pd.notna(name) and str(name).strip():
            return str(name).strip()

    # Final safe fallback without repo_ format.
    return f"Repository {idx}" if idx is not None else str(repo_id)


ranking_df["repository"] = ranking_df["repo_id"].apply(map_repo_id_to_name)
ranking_df["repo_name"] = ranking_df["repository"]

remaining_repo_ids = ranking_df["repo_name"].astype(str).str.match(r"^repo_\d+$", na=False).sum()
print("Repository name mapping completed.")
print("Remaining repo_ IDs:", remaining_repo_ids)
print(ranking_df[["repo_id", "repository"]].head(10))

# Create repo_id on link_df too, but do not use it for metric calculation.
link_df["repo_id"] = [f"repo_{i}" for i in range(len(link_df))]

# Clean numeric metrics from uploaded link-analysis file.
for col in ["pagerank_score", "degree_centrality", "hub_score", "authority_score", "stars", "total_edge_count"]:
    if col in link_df.columns:
        link_df[col] = pd.to_numeric(link_df[col], errors="coerce").fillna(0)

# Keep PageRank and Degree ranking completely separate.
pagerank_rank_df = link_df.sort_values("pagerank_score", ascending=False).reset_index(drop=True).copy()
degree_rank_df = link_df.sort_values("degree_centrality", ascending=False).reset_index(drop=True).copy()

pagerank_rank_df["pagerank_rank"] = np.arange(1, len(pagerank_rank_df) + 1)
degree_rank_df["degree_rank"] = np.arange(1, len(degree_rank_df) + 1)

# A metrics table for display/search only, without combining the metrics.
metrics_df = link_df.copy()
metrics_df = metrics_df.merge(
    pagerank_rank_df[["repository", "pagerank_rank"]],
    on="repository",
    how="left"
).merge(
    degree_rank_df[["repository", "degree_rank"]],
    on="repository",
    how="left"
)

# Technology map from ranking file for compare and similarity graph.

def clean_item(x):
    x = str(x).strip()
    x = x.replace("[", "").replace("]", "")
    x = x.replace("'", "").replace('"', "")
    x = re.sub(r"\s+", "-", x)
    return x.lower().strip()


def parse_technologies(value):
    if pd.isna(value):
        return []
    text = str(value)
    for sep in [",", "+", ";", "/"]:
        text = text.replace(sep, "|")
    parts = [clean_item(p) for p in text.split("|")]
    bad = {"", "nan", "none", "null", "unknown", "[]", "not-specified", "not_specified"}
    return sorted(list(set([p for p in parts if p not in bad])))

ranking_df["tech_list"] = ranking_df["technologies"].apply(parse_technologies)
repo_tech = dict(zip(ranking_df["repo_name"], ranking_df["tech_list"]))

# Add technology count to metrics if available through mapped names.
metrics_df["tech_count"] = metrics_df["repository"].map(lambda x: len(repo_tech.get(x, [])))

# ============================================================
# USE ASSOCIATION/PATTERN FILES DIRECTLY
# ============================================================

# Rules: use the uploaded association_rules file exactly as the source.
for col in ["support", "confidence", "lift", "conviction", "leverage", "CF", "antecedent support", "consequent support"]:
    if col in rules_df.columns:
        rules_df[col] = pd.to_numeric(rules_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

if "rule" not in rules_df.columns:
    if {"antecedents_text", "consequents_text"}.issubset(rules_df.columns):
        rules_df["rule"] = rules_df["antecedents_text"].astype(str) + " -> " + rules_df["consequents_text"].astype(str)
    else:
        rules_df["rule"] = "Rule"

rules_df = rules_df.sort_values(["lift", "confidence", "support"], ascending=False).reset_index(drop=True)

# Itemsets: prefer frequent_technology_combinations if uploaded, otherwise use FP-Growth.
if not freq_combo_df.empty:
    itemsets_df = freq_combo_df.copy()
elif not fpgrowth_df.empty:
    itemsets_df = fpgrowth_df.copy()
elif not apriori_df.empty:
    itemsets_df = apriori_df.copy()
else:
    itemsets_df = pd.DataFrame(columns=["itemsets", "support", "length"])

if "support" in itemsets_df.columns:
    itemsets_df["support"] = pd.to_numeric(itemsets_df["support"], errors="coerce").fillna(0)

if "itemset_text" not in itemsets_df.columns:
    if "itemsets" in itemsets_df.columns:
        itemsets_df["itemset_text"] = itemsets_df["itemsets"].astype(str)
    elif "combination" in itemsets_df.columns:
        itemsets_df["itemset_text"] = itemsets_df["combination"].astype(str)
    else:
        first_col = itemsets_df.columns[0] if len(itemsets_df.columns) else "itemset"
        itemsets_df["itemset_text"] = itemsets_df[first_col].astype(str) if first_col in itemsets_df.columns else ""

if "length" not in itemsets_df.columns:
    itemsets_df["length"] = itemsets_df["itemset_text"].apply(lambda x: len(str(x).split("|")))

# Speed file: use uploaded file directly.
if speed_df.empty:
    speed_df = pd.DataFrame({"algorithm": ["Apriori", "FP-Growth"], "execution_time_seconds": [0, 0]})

# Normalize possible column names for speed.
if "algorithm" not in speed_df.columns:
    speed_df = speed_df.rename(columns={speed_df.columns[0]: "algorithm"})
if "execution_time_seconds" not in speed_df.columns:
    numeric_cols = speed_df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        speed_df = speed_df.rename(columns={numeric_cols[0]: "execution_time_seconds"})
    else:
        speed_df["execution_time_seconds"] = 0
speed_df["execution_time_seconds"] = pd.to_numeric(speed_df["execution_time_seconds"], errors="coerce").fillna(0)

# Pair co-occurrence file: use uploaded file directly.
if not pair_df.empty:
    if "weight" in pair_df.columns:
        pair_df["weight"] = pd.to_numeric(pair_df["weight"], errors="coerce").fillna(0)

# ============================================================
# BUILD REPOSITORY SIMILARITY GRAPH FOR VISUALIZATION ONLY
# PageRank values are taken from repository_link_analysis_results.csv.
# Edges are visualized from technology overlap to make the graph readable.
# ============================================================

top_graph_repos = pagerank_rank_df.head(MAX_GRAPH_REPOS)["repository"].astype(str).tolist()

G = nx.Graph()

for repo in top_graph_repos:
    row = link_df[link_df["repository"] == repo].head(1)
    pr = float(row["pagerank_score"].iloc[0]) if not row.empty else 0
    dc = float(row["degree_centrality"].iloc[0]) if not row.empty else 0
    techs = repo_tech.get(repo, [])
    G.add_node(repo, pagerank_score=pr, degree_centrality=dc, technologies=techs, tech_count=len(techs))

edge_records = []


def build_edges(threshold):
    records = []
    G.remove_edges_from(list(G.edges()))

    for a, b in itertools.combinations(top_graph_repos, 2):
        set_a = set(repo_tech.get(a, []))
        set_b = set(repo_tech.get(b, []))
        if not set_a or not set_b:
            continue

        shared = set_a.intersection(set_b)
        union = set_a.union(set_b)
        score = len(shared) / len(union) if union else 0

        if score >= threshold:
            G.add_edge(a, b, weight=score, shared_count=len(shared), shared_technologies=sorted(list(shared)))
            records.append({
                "source": a,
                "target": b,
                "weight": score,
                "shared_count": len(shared),
                "shared_technologies": " | ".join(sorted(list(shared)))
            })
    return records

edge_records = build_edges(MIN_REPO_JACCARD_EDGE)
if G.number_of_edges() == 0:
    edge_records = build_edges(0.12)

repo_edges_df = pd.DataFrame(edge_records)

# ============================================================
# CLEAN COMPARE DROPDOWN OPTIONS - DEPENDENT PAIR FILTER
# ============================================================
# Important fix:
# Filtering repositories one-by-one is not enough, because two valid repos can
# still have zero similarity together. So we build valid PAIRS first, then the
# second dropdown is filtered based on the first selected repository.
# Result: the user cannot choose a pair that gives 0 shared tech, Very Weak,
# no direct relation, and no association-rule match.

def build_rule_text_blob(df):
    if df is None or df.empty:
        return ""
    possible_cols = ["rule", "antecedents_text", "consequents_text", "antecedents", "consequents"]
    cols = [c for c in possible_cols if c in df.columns]
    if not cols:
        return ""
    return " ".join(
        df[cols].astype(str).fillna("").agg(" ".join, axis=1).tolist()
    ).lower()

rule_text_blob = build_rule_text_blob(rules_df)

def repo_has_nonzero_metrics(repo_name):
    row = metrics_df[metrics_df["repository"].astype(str) == str(repo_name)]
    if row.empty:
        return False
    metric_cols = [
        "pagerank_score",
        "degree_centrality",
        "hub_score",
        "authority_score",
        "total_edge_count",
        "stars",
    ]
    total = 0.0
    for col in metric_cols:
        if col in row.columns:
            total += float(pd.to_numeric(row[col], errors="coerce").fillna(0).iloc[0])
    return total > 0

def shared_techs_between(repo_a, repo_b):
    a = set(repo_tech.get(str(repo_a), []))
    b = set(repo_tech.get(str(repo_b), []))
    return sorted(list(a.intersection(b)))

def parse_rule_items_py(value):
    """Parse association-rule item text into normalized technology items."""
    if pd.isna(value):
        return []
    text = str(value).strip()
    text = text.replace("frozenset", "")
    for ch in ["{", "}", "[", "]", "(", ")", "'", '"']:
        text = text.replace(ch, "")
    # Keep technology labels such as lang:python and topic:ai intact.
    parts = re.split(r"\s*(?:\||,|;)\s*", text)
    bad = {"", "nan", "none", "null"}
    return sorted(set(clean_item(p) for p in parts if clean_item(p) not in bad))

if "antecedent_items" not in rules_df.columns:
    rules_df["antecedent_items"] = rules_df.get("antecedents_text", pd.Series(dtype=str)).apply(parse_rule_items_py)
if "consequent_items" not in rules_df.columns:
    rules_df["consequent_items"] = rules_df.get("consequents_text", pd.Series(dtype=str)).apply(parse_rule_items_py)

def pair_has_exact_association_rule(repo_a, repo_b):
    """
    A clean compare pair must have at least one real association rule where:
    antecedents are contained in repo A technologies and consequents are contained in repo B technologies,
    or the same rule is valid in the opposite direction.
    This prevents dropdown pairs whose displayed metrics would be zero or unrelated.
    """
    tech_a = set(repo_tech.get(str(repo_a), []))
    tech_b = set(repo_tech.get(str(repo_b), []))
    if not tech_a or not tech_b or rules_df.empty:
        return False
    for _, r in rules_df.iterrows():
        ant = set(r.get("antecedent_items", []))
        con = set(r.get("consequent_items", []))
        if not ant or not con:
            continue
        if ant.issubset(tech_a) and con.issubset(tech_b):
            return True
        if ant.issubset(tech_b) and con.issubset(tech_a):
            return True
    return False

# Build valid pair map from the actual graph edges. repo_edges_df already stores
# only useful relationships after the Jaccard threshold, so weak zero pairs are excluded.
valid_compare_pair_map = {}
valid_compare_pair_rows = []

if not repo_edges_df.empty and {"source", "target"}.issubset(repo_edges_df.columns):
    for _, edge in repo_edges_df.iterrows():
        a = str(edge["source"])
        b = str(edge["target"])
        if a == b:
            continue

        weight = float(pd.to_numeric(edge.get("weight", 0), errors="coerce") or 0)
        shared = shared_techs_between(a, b)

        # Strict clean condition:
        # 1) both repos have non-zero ranking metrics
        # 2) pair has a real edge / similarity above threshold
        # 3) pair has shared technologies
        # 4) their shared technologies appear in at least one association rule
        if (
            weight > 0
            and shared
            and repo_has_nonzero_metrics(a)
            and repo_has_nonzero_metrics(b)
            and pair_has_exact_association_rule(a, b)
        ):
            valid_compare_pair_map.setdefault(a, set()).add(b)
            valid_compare_pair_map.setdefault(b, set()).add(a)
            valid_compare_pair_rows.append({
                "repo_a": a,
                "repo_b": b,
                "similarity": weight,
                "shared_count": len(shared),
                "shared_technologies": " | ".join(shared),
            })

# No fallback to weak pairs: Compare dropdown must only include pairs with a real exact association rule.
compare_repo_names = sorted(valid_compare_pair_map.keys())
valid_compare_pair_map = {k: sorted(v) for k, v in valid_compare_pair_map.items() if len(v) > 0}
compare_pairs_df = pd.DataFrame(valid_compare_pair_rows).sort_values(
    ["similarity", "shared_count"], ascending=[False, False]
) if valid_compare_pair_rows else pd.DataFrame()

if len(compare_repo_names) < 2:
    raise ValueError(
        "No clean comparable repository pairs were found. Check repo_edges_df, technologies, and association_rules.csv."
    )

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())
print("Clean compare repositories:", len(compare_repo_names))
print("Clean compare valid pairs:", sum(len(v) for v in valid_compare_pair_map.values()) // 2)
if not compare_pairs_df.empty:
    print("Top clean compare pairs:")
    display(compare_pairs_df.head(10))

# ============================================================
# PLOTLY THEME
# ============================================================

def apply_clean_theme(fig, title=None, height=480):
    fig.update_layout(
        template="plotly_dark",
        title=title,
        height=height,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color=COLOR_TEXT, family="Inter, Segoe UI, Arial"),
        margin=dict(l=35, r=35, t=70, b=35),
        hoverlabel=dict(bgcolor="#020617", font_size=13, font_color=COLOR_TEXT)
    )
    fig.update_xaxes(gridcolor="rgba(255,255,255,0.08)", zerolinecolor="rgba(255,255,255,0.08)")
    fig.update_yaxes(gridcolor="rgba(255,255,255,0.08)", zerolinecolor="rgba(255,255,255,0.08)")
    return fig

# ============================================================
# CHARTS FROM UPLOADED FILES
# ============================================================

plot_itemsets = itemsets_df.copy()
if "support" in plot_itemsets.columns:
    plot_itemsets = plot_itemsets.sort_values("support", ascending=False).head(25)
else:
    plot_itemsets["support"] = 0

fig_itemsets = px.bar(
    plot_itemsets.sort_values("support", ascending=True),
    x="support",
    y="itemset_text",
    orientation="h",
    color="support",
    color_continuous_scale=[COLOR_CYAN, COLOR_PURPLE, COLOR_GREEN],
    hover_data=[c for c in ["length"] if c in plot_itemsets.columns]
)
apply_clean_theme(fig_itemsets, "Frequent Technology Combinations", 560)
fig_itemsets.update_layout(yaxis_title="", xaxis_title="Support")

plot_rules = rules_df.head(300).copy()
if plot_rules.empty:
    fig_rules = go.Figure()
    fig_rules.add_annotation(text="No association rules found in uploaded file.", x=0.5, y=0.5, showarrow=False, font=dict(size=18))
else:
    size_col = "lift" if "lift" in plot_rules.columns else None
    color_col = "CF" if "CF" in plot_rules.columns else "lift"
    fig_rules = px.scatter(
        plot_rules,
        x="support",
        y="confidence",
        size=size_col,
        color=color_col,
        color_continuous_scale=[COLOR_RED, COLOR_AMBER, COLOR_GREEN, COLOR_CYAN, COLOR_PURPLE],
        hover_name="rule",
        hover_data={
            "support": ":.4f",
            "confidence": ":.4f",
            "lift": ":.3f" if "lift" in plot_rules.columns else True,
            "conviction": ":.3f" if "conviction" in plot_rules.columns else True,
            "leverage": ":.4f" if "leverage" in plot_rules.columns else True,
            "CF": ":.3f" if "CF" in plot_rules.columns else True,
        }
    )
apply_clean_theme(fig_rules, "Association Rules from Uploaded File", 560)
fig_rules.update_layout(xaxis_title="Support", yaxis_title="Confidence")

fig_speed = px.bar(
    speed_df,
    x="algorithm",
    y="execution_time_seconds",
    color="algorithm",
    text="execution_time_seconds",
    color_discrete_map={"Apriori": COLOR_CYAN, "FP-Growth": COLOR_PURPLE}
)
apply_clean_theme(fig_speed, "Apriori vs FP-Growth Execution Time", 430)
fig_speed.update_traces(texttemplate="%{text:.4f}s", textposition="outside")
fig_speed.update_layout(showlegend=False, yaxis_title="Seconds", xaxis_title="Algorithm")

# Pair co-occurrence heatmap.
if not pair_df.empty and {"source", "target", "weight"}.issubset(pair_df.columns):
    top_pairs = pair_df.sort_values("weight", ascending=False).head(45).copy()
    pivot_items = sorted(list(set(top_pairs["source"]).union(set(top_pairs["target"]))))[:22]
    heat = pd.DataFrame(0, index=pivot_items, columns=pivot_items)

    for _, row in top_pairs.iterrows():
        s = row["source"]
        t = row["target"]
        w = row["weight"]
        if s in heat.index and t in heat.columns:
            heat.loc[s, t] = w
            heat.loc[t, s] = w

    fig_heatmap = px.imshow(
        heat,
        text_auto=True,
        color_continuous_scale=["#020617", "#0EA5E9", "#22C55E", "#F59E0B"],
        aspect="auto"
    )
    apply_clean_theme(fig_heatmap, "Technology Pair Co-occurrence Heatmap", 650)
else:
    fig_heatmap = go.Figure()
    fig_heatmap.add_annotation(text="No technology_pair_cooccurrence file found.", x=0.5, y=0.5, showarrow=False)
    apply_clean_theme(fig_heatmap, "Technology Pair Co-occurrence Heatmap", 420)

# Separate PageRank chart from uploaded link-analysis file.
top_pr_df = pagerank_rank_df.head(20).copy()
fig_pagerank = px.bar(
    top_pr_df.sort_values("pagerank_score", ascending=True),
    x="pagerank_score",
    y="repository",
    orientation="h",
    color="pagerank_score",
    color_continuous_scale=[COLOR_CYAN, COLOR_PURPLE, COLOR_GREEN],
    hover_data=[c for c in ["degree_centrality", "stars", "language", "total_edge_count"] if c in top_pr_df.columns]
)
apply_clean_theme(fig_pagerank, "Top Repositories by PageRank", 620)
fig_pagerank.update_layout(xaxis_title="PageRank Score", yaxis_title="")

# Separate Degree Centrality chart from uploaded link-analysis file.
top_degree_df = degree_rank_df.head(20).copy()
fig_degree = px.bar(
    top_degree_df.sort_values("degree_centrality", ascending=True),
    x="degree_centrality",
    y="repository",
    orientation="h",
    color="degree_centrality",
    color_continuous_scale=[COLOR_CYAN, COLOR_GREEN, COLOR_AMBER],
    hover_data=[c for c in ["pagerank_score", "stars", "language", "total_edge_count"] if c in top_degree_df.columns]
)
apply_clean_theme(fig_degree, "Top Repositories by Degree Centrality", 620)
fig_degree.update_layout(xaxis_title="Degree Centrality", yaxis_title="")

# Metric relationship scatter is displayed in the Overview page.
fig_metric_scatter = px.scatter(
    metrics_df,
    x="degree_centrality",
    y="pagerank_score",
    size="tech_count",
    color="pagerank_score",
    color_continuous_scale=[COLOR_CYAN, COLOR_PURPLE, COLOR_AMBER],
    hover_name="repository",
    hover_data=[c for c in ["pagerank_rank", "degree_rank", "hub_score", "authority_score", "stars", "language"] if c in metrics_df.columns]
)
apply_clean_theme(fig_metric_scatter, "PageRank vs Degree Centrality Comparison", 560)
fig_metric_scatter.update_layout(xaxis_title="Degree Centrality", yaxis_title="PageRank Score")

# Network graph by PageRank, names only.

def short_repo_label(name, max_len=24):
    name = str(name)
    label = name.split("/", 1)[1] if "/" in name else name
    return label[:max_len] + "..." if len(label) > max_len else label

if G.number_of_nodes() > 0:
    pos = nx.spring_layout(G, seed=42, k=1.45, iterations=260, weight="weight")

    edge_x, edge_y = [], []
    for u, v, data in G.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    node_x, node_y, node_text, node_size, node_color, node_labels = [], [], [], [], [], []
    max_pr = max([G.nodes[n].get("pagerank_score", 0) for n in G.nodes()]) or 1

    for node in G.nodes():
        x, y = pos[node]
        pr = float(G.nodes[node].get("pagerank_score", 0))
        dc = float(G.nodes[node].get("degree_centrality", 0))
        techs = G.nodes[node].get("technologies", [])

        node_x.append(x)
        node_y.append(y)
        node_size.append(12 + 42 * (pr / max_pr))
        node_color.append(pr)
        node_labels.append(short_repo_label(node))
        node_text.append(
            f"<b>{node}</b><br>"
            f"PageRank: {pr:.8f}<br>"
            f"Degree Centrality: {dc:.6f}<br>"
            f"Technologies: {len(techs)}"
        )

    fig_network = go.Figure()
    fig_network.add_trace(go.Scattergl(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(width=0.8, color="rgba(148,163,184,0.20)"),
        hoverinfo="none",
        name="Similarity Links"
    ))
    fig_network.add_trace(go.Scattergl(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=node_labels,
        textposition="top center",
        textfont=dict(size=10, color="rgba(229,231,235,0.90)"),
        marker=dict(
            size=node_size,
            color=node_color,
            colorscale=[[0, COLOR_CYAN], [0.50, COLOR_PURPLE], [1, COLOR_GREEN]],
            showscale=True,
            colorbar=dict(title="PageRank"),
            line=dict(width=1.2, color="rgba(255,255,255,0.75)")
        ),
        hovertext=node_text,
        hovertemplate="%{hovertext}<extra></extra>",
        name="Repositories"
    ))
    fig_network.update_layout(
        title="Network Graph by PageRank",
        height=780,
        showlegend=False,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color=COLOR_TEXT),
        margin=dict(l=10, r=10, t=70, b=10),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False)
    )
else:
    fig_network = go.Figure()
    fig_network.add_annotation(text="No graph nodes available.", x=0.5, y=0.5, showarrow=False)

# ============================================================
# JSON + HTML HELPERS
# ============================================================

def clean_json_value(x):
    if isinstance(x, (set, frozenset, list, tuple)):
        return " | ".join(sorted([str(i) for i in x]))
    if isinstance(x, dict):
        return {str(k): clean_json_value(v) for k, v in x.items()}
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return None if np.isnan(x) or np.isinf(x) else float(x)
    if isinstance(x, float):
        return None if math.isnan(x) or math.isinf(x) else x
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    if isinstance(x, (str, int, bool, type(None))):
        return x
    return str(x)


def df_to_records_safe(df):
    if df is None or df.empty:
        return []
    clean_df = df.copy().replace([np.inf, -np.inf], np.nan)
    for col in clean_df.columns:
        clean_df[col] = clean_df[col].apply(clean_json_value)
    clean_df = clean_df.where(pd.notnull(clean_df), None)
    return clean_df.to_dict(orient="records")


def fig_html(fig):
    return pio.to_html(
        fig,
        full_html=False,
        include_plotlyjs=False,
        config={"responsive": True, "displaylogo": False, "modeBarButtonsToRemove": ["lasso2d", "select2d"]}
    )

fig_itemsets_html = fig_html(fig_itemsets)
fig_rules_html = fig_html(fig_rules)
fig_speed_html = fig_html(fig_speed)
fig_heatmap_html = fig_html(fig_heatmap)
fig_network_html = fig_html(fig_network)
fig_pagerank_html = fig_html(fig_pagerank)
fig_degree_html = fig_html(fig_degree)
fig_metric_scatter_html = fig_html(fig_metric_scatter)
# Separate HTML copy so Plotly div IDs are not duplicated between Overview and Network pages.
fig_metric_scatter_network_html = fig_html(fig_metric_scatter)

# Data for JS interactions.
metrics_records = df_to_records_safe(metrics_df)
edge_records_js = df_to_records_safe(repo_edges_df)
rules_records = df_to_records_safe(rules_df)
repo_tech_json_safe = {str(k): [str(i) for i in v] for k, v in repo_tech.items()}

metrics_json = json.dumps(metrics_records, ensure_ascii=False)
edge_json = json.dumps(edge_records_js, ensure_ascii=False)
rules_json = json.dumps(rules_records, ensure_ascii=False)
repo_tech_json = json.dumps(repo_tech_json_safe, ensure_ascii=False)
valid_compare_pair_map_json = json.dumps(valid_compare_pair_map, ensure_ascii=False)

# ============================================================
# TABLES AND OVERVIEW VALUES
# ============================================================

repo_options = "\n".join([f'<option value="{r}">{r}</option>' for r in compare_repo_names])

top_pagerank_repo = pagerank_rank_df.iloc[0]["repository"] if not pagerank_rank_df.empty else "N/A"
top_pagerank_value = pagerank_rank_df.iloc[0]["pagerank_score"] if not pagerank_rank_df.empty else 0

top_degree_repo = degree_rank_df.iloc[0]["repository"] if not degree_rank_df.empty else "N/A"
top_degree_value = degree_rank_df.iloc[0]["degree_centrality"] if not degree_rank_df.empty else 0

best_rule = rules_df.iloc[0]["rule"] if not rules_df.empty and "rule" in rules_df.columns else "N/A"
best_rule_lift = rules_df.iloc[0]["lift"] if not rules_df.empty and "lift" in rules_df.columns else 0

# Rules table from uploaded file.
rules_table_rows = ""
if rules_df.empty:
    rules_table_rows = '<tr><td colspan="7">No association rules found.</td></tr>'
else:
    for _, row in rules_df.head(TOP_ROWS).iterrows():
        rules_table_rows += f"""
        <tr>
            <td>{row.get('rule', '')}</td>
            <td>{float(row.get('support', 0)):.4f}</td>
            <td>{float(row.get('confidence', 0)):.4f}</td>
            <td>{float(row.get('lift', 0)):.3f}</td>
            <td>{float(row.get('conviction', 0)):.3f}</td>
            <td>{float(row.get('leverage', 0)):.4f}</td>
            <td>{float(row.get('CF', 0)):.3f}</td>
        </tr>
        """

# PageRank table.
pagerank_table_rows = ""
for _, row in pagerank_rank_df.head(TOP_ROWS).iterrows():
    pagerank_table_rows += f"""
    <tr>
        <td>{int(row.get('pagerank_rank', 0))}</td>
        <td>{row.get('repository', '')}</td>
        <td>{float(row.get('pagerank_score', 0)):.8f}</td>
        <td>{float(row.get('degree_centrality', 0)):.6f}</td>
        <td>{int(row.get('stars', 0)) if 'stars' in row else 0}</td>
        <td>{row.get('language', '') if 'language' in row else ''}</td>
    </tr>
    """

# Degree table.
degree_table_rows = ""
for _, row in degree_rank_df.head(TOP_ROWS).iterrows():
    degree_table_rows += f"""
    <tr>
        <td>{int(row.get('degree_rank', 0))}</td>
        <td>{row.get('repository', '')}</td>
        <td>{float(row.get('degree_centrality', 0)):.6f}</td>
        <td>{float(row.get('pagerank_score', 0)):.8f}</td>
        <td>{int(row.get('total_edge_count', 0)) if 'total_edge_count' in row else 0}</td>
        <td>{row.get('language', '') if 'language' in row else ''}</td>
    </tr>
    """

# ============================================================
# FINAL HTML DASHBOARD
# ============================================================

html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Data Mining Dashboard</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>
* {{ box-sizing: border-box; }}
body {{
    margin: 0;
    font-family: Inter, Segoe UI, Arial, sans-serif;
    background: radial-gradient(circle at top left, rgba(56,189,248,0.13), transparent 28%),
                radial-gradient(circle at top right, rgba(139,92,246,0.13), transparent 28%),
                #0B1120;
    color: #E5E7EB;
}}
.sidebar {{
    position: fixed;
    top: 0;
    left: 0;
    width: 292px;
    height: 100vh;
    background: rgba(2,6,23,0.94);
    backdrop-filter: blur(18px);
    border-right: 1px solid rgba(255,255,255,0.08);
    padding: 26px 18px;
    z-index: 10;
}}
.logo {{
    font-size: 24px;
    font-weight: 900;
    background: linear-gradient(90deg, #38BDF8, #8B5CF6, #22C55E);
    -webkit-background-clip: text;
    color: transparent;
    margin-bottom: 8px;
}}
.subtitle {{ color: #94A3B8; font-size: 13px; line-height: 1.5; margin-bottom: 25px; }}
.nav-btn {{
    width: 100%;
    padding: 13px 15px;
    margin-bottom: 10px;
    border-radius: 14px;
    border: 1px solid rgba(255,255,255,0.08);
    background: rgba(15,23,42,0.8);
    color: #CBD5E1;
    cursor: pointer;
    text-align: left;
    font-size: 14px;
    transition: 0.2s ease;
}}
.nav-btn:hover, .nav-btn.active {{
    background: linear-gradient(135deg, rgba(14,165,233,0.95), rgba(139,92,246,0.95));
    color: white;
    transform: translateX(4px);
}}
.main {{ margin-left: 292px; padding: 30px; }}
.header {{ display: flex; align-items: center; justify-content: space-between; margin-bottom: 24px; }}
.header h1 {{ margin: 0; font-size: 34px; font-weight: 900; }}
.header p {{ color: #94A3B8; margin-top: 8px; }}
.badge {{
    padding: 12px 16px;
    border-radius: 999px;
    color: #38BDF8;
    background: rgba(56,189,248,0.12);
    border: 1px solid rgba(56,189,248,0.35);
    font-weight: 700;
}}
.cards {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 18px; margin-bottom: 22px; }}
.card {{
    background: linear-gradient(145deg, rgba(17,24,39,0.96), rgba(15,23,42,0.96));
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 22px;
    padding: 20px;
    box-shadow: 0 20px 60px rgba(0,0,0,0.32);
}}
.card .label {{ color: #94A3B8; font-size: 13px; }}
.card .value {{ font-size: 31px; font-weight: 900; margin-top: 8px; color: #38BDF8; }}
.card .hint {{ color: #64748B; font-size: 12px; margin-top: 6px; }}
.section {{ display: none; }}
.section.active {{ display: block; }}
.grid-2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 18px; }}
.grid-3 {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 18px; }}
.panel {{
    background: rgba(17,24,39,0.86);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 22px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 0 16px 55px rgba(0,0,0,0.26);
}}
.panel h2 {{ margin-top: 0; font-size: 20px; }}
.insight {{
    background: linear-gradient(135deg, rgba(56,189,248,0.08), rgba(139,92,246,0.08));
    border-left: 4px solid #38BDF8;
    border-radius: 15px;
    padding: 15px;
    color: #CBD5E1;
    line-height: 1.7;
    margin-top: 14px;
}}
.select-grid {{ display: grid; grid-template-columns: 1fr 1fr auto; gap: 14px; align-items: end; }}
select, input {{
    width: 100%;
    padding: 13px 14px;
    border-radius: 14px;
    border: 1px solid rgba(255,255,255,0.12);
    background: #020617;
    color: #E5E7EB;
    outline: none;
}}
.action-btn {{
    padding: 13px 18px;
    border-radius: 14px;
    border: 0;
    background: linear-gradient(135deg, #0EA5E9, #8B5CF6);
    color: white;
    font-weight: 800;
    cursor: pointer;
    min-width: 170px;
}}
.result-box {{
    background: rgba(15,23,42,0.9);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 18px;
    padding: 18px;
    margin-top: 16px;
    line-height: 1.8;
}}
.metric-pill {{
    display: inline-block;
    padding: 8px 12px;
    margin: 5px 5px 5px 0;
    border-radius: 999px;
    background: rgba(56,189,248,0.12);
    border: 1px solid rgba(56,189,248,0.35);
    color: #38BDF8;
    font-weight: 800;
}}
.strength-bar {{ width: 100%; height: 16px; background: #020617; border-radius: 999px; overflow: hidden; margin: 12px 0; }}
.strength-fill {{ height: 100%; background: linear-gradient(90deg, #EF4444, #F59E0B, #22C55E, #38BDF8); }}
table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
th {{ text-align: left; color: #38BDF8; background: rgba(2,6,23,0.75); padding: 12px; border-bottom: 1px solid rgba(255,255,255,0.1); }}
td {{ padding: 11px; border-bottom: 1px solid rgba(255,255,255,0.06); color: #CBD5E1; }}
tr:hover {{ background: rgba(56,189,248,0.06); }}
@media(max-width: 1100px) {{
    .cards, .grid-2, .grid-3, .select-grid {{ grid-template-columns: 1fr; }}
    .sidebar {{ position: relative; width: 100%; height: auto; }}
    .main {{ margin-left: 0; }}
}}
</style>
</head>
<body>

<div class="sidebar">
    <div class="logo">GitHub Data Mining</div>
    <div class="subtitle">Professional local dashboard using the uploaded result files.</div>

    <button class="nav-btn active" onclick="showSection('overview', this)">Overview</button>
    <button class="nav-btn" onclick="showSection('patterns', this)">Pattern Mining</button>
    <button class="nav-btn" onclick="showSection('rules', this)">Association Rules</button>
    <button class="nav-btn" onclick="showSection('pagerank', this)">PageRank Ranking</button>
    <button class="nav-btn" onclick="showSection('degree', this)">Degree Centrality Ranking</button>
    <button class="nav-btn" onclick="showSection('graph', this)">Network Graph by PageRank</button>
    <button class="nav-btn" onclick="showSection('compare', this)">Compare Repositories</button>
</div>

<div class="main">
    <div class="header">
        <div>
            <h1>Professional Data Mining Dashboard</h1>
            <p>Pattern mining, association rules, repository link analysis, PageRank, and Degree Centrality.</p>
        </div>
        <div class="badge">Local HTML Dashboard</div>
    </div>

    <div class="cards">
        <div class="card">
            <div class="label">Repositories</div>
            <div class="value">{len(link_df)}</div>
            <div class="hint">From repository link analysis file</div>
        </div>
        <div class="card">
            <div class="label">Association Rules</div>
            <div class="value">{len(rules_df)}</div>
            <div class="hint">From uploaded association rules file</div>
        </div>
        <div class="card">
            <div class="label">Top PageRank Repo</div>
            <div class="value" style="font-size:18px;">{top_pagerank_repo}</div>
            <div class="hint">PageRank: {top_pagerank_value:.8f}</div>
        </div>
        <div class="card">
            <div class="label">Top Degree Repo</div>
            <div class="value" style="font-size:18px;">{top_degree_repo}</div>
            <div class="hint">Degree: {top_degree_value:.6f}</div>
        </div>
    </div>

    <div id="overview" class="section active">
        <div class="grid-2">
            <div class="panel">
                <h2>Project Overview</h2>
                <div class="insight">
                    This dashboard uses your exported CSV files directly. Pattern mining and association rules are loaded from the association result files. Repository names are mapped only for readable display. PageRank and Degree Centrality are treated as two independent ranking methods.
                </div>
            </div>
            <div class="panel">
                <h2>Main Findings</h2>
                <div class="insight">
                    <b>Top PageRank repository:</b> {top_pagerank_repo}<br>
                    <b>PageRank score:</b> {top_pagerank_value:.8f}<br><br>
                    <b>Top Degree Centrality repository:</b> {top_degree_repo}<br>
                    <b>Degree Centrality score:</b> {top_degree_value:.6f}<br><br>
                    <b>Best association rule by lift:</b> {best_rule}<br>
                    <b>Lift:</b> {best_rule_lift:.3f}
                </div>
            </div>
        </div>

        <div class="panel">
            <h2>PageRank vs Degree Centrality</h2>
            {fig_metric_scatter_html}
            <div class="insight">
                This visualization compares the PageRank score and Degree Centrality score for each repository using the uploaded link-analysis file. PageRank and Degree Centrality are kept as separate metrics.
            </div>
        </div>

        <div class="grid-2">
            <div class="panel">{fig_speed_html}</div>
            <div class="panel">{fig_heatmap_html}</div>
        </div>
    </div>

    <div id="patterns" class="section">
        <div class="panel">
            <h2>Frequent Technology Combinations</h2>
            {fig_itemsets_html}
            <div class="insight">This chart is loaded from your frequent itemsets / frequent combinations CSV files.</div>
        </div>

    </div>

    <div id="rules" class="section">
        <div class="panel">
            <h2>Association Rules</h2>
            {fig_rules_html}
            <div class="insight">These rules are loaded from your uploaded association_rules.csv file, not recalculated inside this dashboard.</div>
        </div>
        <div class="panel">
            <h2>Top Association Rules Table</h2>
            <table>
                <thead>
                    <tr>
                        <th>Rule</th>
                        <th>Support</th>
                        <th>Confidence</th>
                        <th>Lift</th>
                        <th>Conviction</th>
                        <th>Leverage</th>
                        <th>CF</th>
                    </tr>
                </thead>
                <tbody>{rules_table_rows}</tbody>
            </table>
        </div>
    </div>

    <div id="pagerank" class="section">
        <div class="panel">
            <h2>PageRank Ranking</h2>
            {fig_pagerank_html}
            <div class="insight">This page ranks repositories only by the pagerank_score column from repository_link_analysis_results.csv.</div>
        </div>
        <div class="panel">
            <h2>Top Repositories by PageRank</h2>
            <input id="pagerankSearch" onkeyup="filterSpecificTable('pagerankSearch', 'pagerankTable')" placeholder="Search PageRank table..." />
            <br><br>
            <table id="pagerankTable">
                <thead>
                    <tr>
                        <th>Rank</th>
                        <th>Repository</th>
                        <th>PageRank</th>
                        <th>Degree Centrality</th>
                        <th>Stars</th>
                        <th>Language</th>
                    </tr>
                </thead>
                <tbody>{pagerank_table_rows}</tbody>
            </table>
        </div>
    </div>

    <div id="degree" class="section">
        <div class="panel">
            <h2>Degree Centrality Ranking</h2>
            {fig_degree_html}
            <div class="insight">This page ranks repositories only by the degree_centrality column from repository_link_analysis_results.csv.</div>
        </div>
        <div class="panel">
            <h2>Top Repositories by Degree Centrality</h2>
            <input id="degreeSearch" onkeyup="filterSpecificTable('degreeSearch', 'degreeTable')" placeholder="Search Degree table..." />
            <br><br>
            <table id="degreeTable">
                <thead>
                    <tr>
                        <th>Rank</th>
                        <th>Repository</th>
                        <th>Degree Centrality</th>
                        <th>PageRank</th>
                        <th>Total Edges</th>
                        <th>Language</th>
                    </tr>
                </thead>
                <tbody>{degree_table_rows}</tbody>
            </table>
        </div>
    </div>

    <div id="graph" class="section">
        <div class="panel">
            <h2>Network Graph by PageRank</h2>
            {fig_network_html}
            <div class="insight">Nodes are repository names. Node size and color represent PageRank from the uploaded link-analysis file. Edges are drawn from shared technologies only to make the graph readable.</div>
        </div>

        <div class="panel">
            <h2>PageRank vs Degree Centrality</h2>
            {fig_metric_scatter_network_html}
            <div class="insight">
                This graph is also included here to compare each repository's PageRank influence against its direct connectivity while viewing the network analysis page.
            </div>
        </div>

    </div>

    <div id="compare" class="section">
        <div class="panel">
            <h2>Compare Two Repositories</h2>
            <p style="color:#94A3B8;">Choose repositories from dropdown menus. No manual typing required.</p>
            <div class="select-grid">
                <div>
                    <label>Repository A</label>
                    <select id="repoA" onchange="updateRepoBDropdown()"></select>
                </div>
                <div>
                    <label>Repository B</label>
                    <select id="repoB"></select>
                </div>
                <button class="action-btn" onclick="compareRepos()">Analyze</button>
            </div>
            <div id="compareResult" class="result-box">Select two repositories and click Analyze.</div>
            <div id="compareGraph" style="height:560px; margin-top:20px;"></div>
        </div>
    </div>
</div>

<script>
const metricsData = {metrics_json};
const edgeData = {edge_json};
const rulesData = {rules_json};
const repoTechMap = {repo_tech_json};
const validComparePairs = {valid_compare_pair_map_json};

function showSection(id, btn) {{
    document.querySelectorAll('.section').forEach(s => s.classList.remove('active'));
    document.getElementById(id).classList.add('active');
    document.querySelectorAll('.sidebar .nav-btn').forEach(b => b.classList.remove('active'));
    btn.classList.add('active');
}}

function filterSpecificTable(inputId, tableId) {{
    const q = document.getElementById(inputId).value.toLowerCase();
    document.querySelectorAll(`#${{tableId}} tbody tr`).forEach(row => {{
        row.style.display = row.innerText.toLowerCase().includes(q) ? '' : 'none';
    }});
}}

function getRepo(repoName) {{
    return metricsData.find(r => r.repository === repoName) || {{ repository: repoName }};
}}

function getDirectEdge(a, b) {{
    return edgeData.find(e => (e.source === a && e.target === b) || (e.source === b && e.target === a));
}}

function jaccard(aList, bList) {{
    const A = new Set(aList || []);
    const B = new Set(bList || []);
    const inter = [...A].filter(x => B.has(x));
    const union = new Set([...A, ...B]);
    return {{ score: union.size ? inter.length / union.size : 0, shared: inter, unionSize: union.size }};
}}

function strengthLabel(score) {{
    if (score >= 0.50) return ['Strong', '#22C55E'];
    if (score >= 0.28) return ['Moderate', '#F59E0B'];
    if (score >= 0.12) return ['Weak', '#38BDF8'];
    return ['Very Weak', '#EF4444'];
}}

function normalizeTechItem(x) {{
    return String(x || '')
        .trim()
        .replace(/[\[\]{{}}()'"]/g, '')
        .replace(/\s+/g, '-')
        .toLowerCase();
}}

function splitRuleItems(value) {{
    if (Array.isArray(value)) return value.map(normalizeTechItem).filter(Boolean);
    const text = String(value || '')
        .replace(/frozenset/g, '')
        .replace(/[\[\]{{}}()'"]/g, '');
    return [...new Set(text.split(/\s*(?:\||,|;)\s*/).map(normalizeTechItem).filter(x => x && !['nan','none','null'].includes(x)))];
}}

const transactionSets = Object.values(repoTechMap)
    .filter(items => Array.isArray(items) && items.length)
    .map(items => new Set(items.map(normalizeTechItem)));

function setContainsAll(setObj, items) {{
    return items.every(item => setObj.has(normalizeTechItem(item)));
}}

function calculateRuleMetrics(antecedents, consequents) {{
    const ant = [...new Set((antecedents || []).map(normalizeTechItem).filter(Boolean))];
    const con = [...new Set((consequents || []).map(normalizeTechItem).filter(Boolean))];
    const N = transactionSets.length;
    if (!N || !ant.length || !con.length) return null;

    let countA = 0, countB = 0, countAB = 0;
    transactionSets.forEach(t => {{
        const hasA = setContainsAll(t, ant);
        const hasB = setContainsAll(t, con);
        if (hasA) countA += 1;
        if (hasB) countB += 1;
        if (hasA && hasB) countAB += 1;
    }});

    const supportA = countA / N;
    const supportB = countB / N;
    const supportAB = countAB / N;
    const confidence = supportA > 0 ? supportAB / supportA : 0;
    const lift = supportB > 0 ? confidence / supportB : 0;
    const leverage = supportAB - (supportA * supportB);
    const conviction = confidence >= 1 ? Infinity : ((1 - supportB) / Math.max(1 - confidence, 1e-12));
    let CF = 0;
    if (confidence >= supportB) {{
        CF = (1 - supportB) > 0 ? (confidence - supportB) / (1 - supportB) : 0;
    }} else {{
        CF = supportB > 0 ? (confidence - supportB) / supportB : 0;
    }}

    return {{
        support: supportAB,
        antecedent_support: supportA,
        consequent_support: supportB,
        confidence,
        lift,
        leverage,
        conviction,
        CF,
        countA,
        countB,
        countAB,
        total: N
    }};
}}

function getRuleItems(rule) {{
    let ant = splitRuleItems(rule.antecedent_items || rule.antecedents_text || '');
    let con = splitRuleItems(rule.consequent_items || rule.consequents_text || '');
    if ((!ant.length || !con.length) && rule.rule) {{
        const parts = String(rule.rule).split(/→|->|=>/);
        if (parts.length >= 2) {{
            ant = splitRuleItems(parts[0]);
            con = splitRuleItems(parts.slice(1).join(' '));
        }}
    }}
    return {{ ant, con }};
}}

function findMatchingRules(techA, techB) {{
    const A = new Set((techA || []).map(normalizeTechItem));
    const B = new Set((techB || []).map(normalizeTechItem));
    const matches = [];

    rulesData.forEach(rule => {{
        const {{ ant, con }} = getRuleItems(rule);
        if (!ant.length || !con.length) return;

        const forward = ant.every(x => A.has(x)) && con.every(x => B.has(x));
        const reverse = ant.every(x => B.has(x)) && con.every(x => A.has(x));
        if (!forward && !reverse) return;

        const metrics = forward ? calculateRuleMetrics(ant, con) : calculateRuleMetrics(con, ant);
        if (!metrics || metrics.support <= 0 || metrics.confidence <= 0 || metrics.lift <= 0) return;

        const displayAnt = forward ? ant : con;
        const displayCon = forward ? con : ant;
        matches.push({{
            ...rule,
            ...metrics,
            calculated_rule: `${{displayAnt.join(', ')}} → ${{displayCon.join(', ')}}`,
            direction: forward ? 'A to B' : 'B to A'
        }});
    }});

    matches.sort((a, b) =>
        Number(b.lift || 0) - Number(a.lift || 0) ||
        Number(b.confidence || 0) - Number(a.confidence || 0) ||
        Number(b.support || 0) - Number(a.support || 0)
    );
    return matches;
}}

function fillSelect(selectId, values, selectedValue=null) {{
    const select = document.getElementById(selectId);
    select.innerHTML = '';
    values.forEach(v => {{
        const option = document.createElement('option');
        option.value = v;
        option.textContent = v;
        select.appendChild(option);
    }});
    if (selectedValue && values.includes(selectedValue)) select.value = selectedValue;
}}

function updateRepoBDropdown() {{
    const a = document.getElementById('repoA').value;
    const bSelect = document.getElementById('repoB');
    const currentB = bSelect.value;
    const related = (validComparePairs[a] || []).filter(x => x !== a).sort();
    fillSelect('repoB', related, currentB);
    if (!related.includes(currentB) && related.length) bSelect.value = related[0];
    compareRepos();
}}

function initializeCompareDropdowns() {{
    const repoAValues = Object.keys(validComparePairs).sort();
    fillSelect('repoA', repoAValues);
    if (repoAValues.length) {{
        document.getElementById('repoA').value = repoAValues[0];
        updateRepoBDropdown();
    }}
}}

function compareRepos() {{
    const a = document.getElementById('repoA').value;
    const b = document.getElementById('repoB').value;
    const box = document.getElementById('compareResult');

    if (!a || !b) {{
        box.innerHTML = 'No clean related repositories are available for this selection.';
        Plotly.purge('compareGraph');
        return;
    }}

    if (a === b) {{
        box.innerHTML = 'Please select two different repositories.';
        return;
    }}

    const repoA = getRepo(a);
    const repoB = getRepo(b);
    const techA = repoTechMap[a] || [];
    const techB = repoTechMap[b] || [];
    const jac = jaccard(techA, techB);
    const direct = getDirectEdge(a, b);
    const graphScore = Math.max(jac.score, direct ? Number(direct.weight || 0) : 0);
    const label = strengthLabel(graphScore);
    const matchingRules = findMatchingRules(techA, techB);
    const bestRule = matchingRules.length ? matchingRules[0] : null;

    const sharedText = jac.shared.length ? jac.shared.slice(0, 35).join(', ') : 'No shared technologies.';
    let ruleText = 'No matching association rule was found for their shared technologies.';
    if (bestRule) {{
        ruleText = `<b>Best Matching Association Rule:</b><br>${{bestRule.calculated_rule || bestRule.rule}}<br>Direction: ${{bestRule.direction || 'A to B'}}<br>Support: ${{Number(bestRule.support || 0).toFixed(4)}} | Confidence: ${{Number(bestRule.confidence || 0).toFixed(4)}} | Lift: ${{Number(bestRule.lift || 0).toFixed(3)}} | CF: ${{Number(bestRule.CF || 0).toFixed(3)}} | Leverage: ${{Number(bestRule.leverage || 0).toFixed(4)}}<br><span class="muted">Calculated from ${{bestRule.total || 0}} repository transactions: support(A∪B), confidence=support(A∪B)/support(A), lift=confidence/support(B).</span>`;
    }}

    box.innerHTML = `
        <span class="metric-pill" style="color:${{label[1]}}; border-color:${{label[1]}};">${{label[0]}} Relationship</span>
        <span class="metric-pill">${{(graphScore * 100).toFixed(2)}}% Similarity</span>
        <span class="metric-pill">${{jac.shared.length}} Shared Technologies</span>
        <span class="metric-pill">Direct Edge: ${{direct ? 'Yes' : 'No'}}</span>
        <div class="strength-bar"><div class="strength-fill" style="width:${{Math.min(graphScore * 100, 100)}}%"></div></div>
        <br><b>Repository A:</b> ${{a}}
        <br>PageRank: ${{Number(repoA.pagerank_score || 0).toFixed(8)}} | PageRank Rank: ${{repoA.pagerank_rank || 'N/A'}}
        <br>Degree Centrality: ${{Number(repoA.degree_centrality || 0).toFixed(6)}} | Degree Rank: ${{repoA.degree_rank || 'N/A'}}
        <br><br><b>Repository B:</b> ${{b}}
        <br>PageRank: ${{Number(repoB.pagerank_score || 0).toFixed(8)}} | PageRank Rank: ${{repoB.pagerank_rank || 'N/A'}}
        <br>Degree Centrality: ${{Number(repoB.degree_centrality || 0).toFixed(6)}} | Degree Rank: ${{repoB.degree_rank || 'N/A'}}
        <br><br><b>Shared Technologies:</b><br>${{sharedText}}
        <br><br>${{ruleText}}
    `;

    drawCompareGraph(a, b, jac.shared, direct);
}}

function drawCompareGraph(a, b, sharedTechs, direct) {{
    const nodes = [{{ id: a, type: 'repo' }}, {{ id: b, type: 'repo' }}, ...sharedTechs.slice(0, 18).map(t => ({{ id: t, type: 'tech' }}))];
    const angleStep = 2 * Math.PI / Math.max(nodes.length, 1);
    const x = [], y = [], text = [], colors = [], sizes = [];

    nodes.forEach((n, i) => {{
        if (n.type === 'repo' && n.id === a) {{ x.push(-0.75); y.push(0); }}
        else if (n.type === 'repo' && n.id === b) {{ x.push(0.75); y.push(0); }}
        else {{ const angle = i * angleStep; x.push(Math.cos(angle) * 1.15); y.push(Math.sin(angle) * 1.15); }}
        text.push(n.id);
        colors.push(n.type === 'repo' ? (n.id === a ? '#22C55E' : '#F59E0B') : '#38BDF8');
        sizes.push(n.type === 'repo' ? 34 : 18);
    }});

    const edgeX = [], edgeY = [];
    function addEdge(i, j) {{ edgeX.push(x[i], x[j], null); edgeY.push(y[i], y[j], null); }}
    if (direct) addEdge(0, 1);
    for (let i = 2; i < nodes.length; i++) {{ addEdge(0, i); addEdge(1, i); }}

    const edgeTrace = {{ x: edgeX, y: edgeY, mode: 'lines', type: 'scatter', line: {{ color: 'rgba(148,163,184,0.45)', width: 2 }}, hoverinfo: 'none' }};
    const nodeTrace = {{ x: x, y: y, mode: 'markers+text', type: 'scatter', text: text, textposition: 'top center', marker: {{ size: sizes, color: colors, line: {{ color: 'white', width: 1 }} }}, hovertemplate: '<b>%{{text}}</b><extra></extra>' }};
    const layout = {{ title: 'Focused Repository Relationship Graph', paper_bgcolor: 'rgba(0,0,0,0)', plot_bgcolor: 'rgba(0,0,0,0)', font: {{ color: '#E5E7EB' }}, height: 540, showlegend: false, margin: {{ l: 10, r: 10, t: 60, b: 10 }}, xaxis: {{ visible: false }}, yaxis: {{ visible: false }} }};
    Plotly.newPlot('compareGraph', [edgeTrace, nodeTrace], layout, {{ responsive: true, displaylogo: false }});
}}

window.addEventListener('load', () => {{
    initializeCompareDropdowns();
}});
</script>
</body>
</html>
"""

# ============================================================
# SAVE OUTPUTS
# ============================================================

with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)

# Save separated rankings for verification.
pagerank_rank_df.to_csv("SEPARATE_top_repositories_by_pagerank.csv", index=False)
degree_rank_df.to_csv("SEPARATE_top_repositories_by_degree_centrality.csv", index=False)
metrics_df.to_csv("SEPARATE_repository_metrics_mapped_names.csv", index=False)

print("DONE")
print("Dashboard saved as:", OUTPUT_HTML)
print("PageRank ranking file: SEPARATE_top_repositories_by_pagerank.csv")
print("Degree Centrality ranking file: SEPARATE_top_repositories_by_degree_centrality.csv")
print("Mapped metrics file: SEPARATE_repository_metrics_mapped_names.csv")
print("Important: PageRank and Degree Centrality are separate. No combined final rank is used.")

files.download(OUTPUT_HTML)


Loaded files:
- Ranking: top_repositories_ranking.csv
- Link Analysis: repository_link_analysis_results.csv
- Association Rules: association_rules.csv
- Strong Rules: strong_association_rules.csv
- Apriori: apriori_frequent_itemsets.csv
- FP-Growth: fpgrowth_frequent_itemsets.csv
- Frequent Combinations: frequent_technology_combinations.csv
- Speed: apriori_vs_fpgrowth_speed_comparison.csv
- Pair Co-occurrence: technology_pair_cooccurrence.csv
Repository name mapping completed.
Remaining repo_ IDs: 0
     repo_id                         repository
0   repo_172                    langgenius/dify
1   repo_738             rodrigo-brito/ninjabot
2   repo_287                     go-gitea/gitea
3   repo_328              trailofbits/manticore
4   repo_318            tensorlayer/TensorLayer
5     repo_3                      arc53/DocsGPT
6   repo_423  responsively-org/responsively-app
7    repo_30                      moonrepo/moon
8    repo_59                    pycaret/pycaret
9  repo_3040  

,repo_a,repo_b,similarity,shared_count,shared_technologies
26,pulumi/pulumi,aws-samples/bedrock-chat,0.545455,6,lang:python | topic:data-science | topic:deep-...
5,GitHubDaily/GitHubDaily,obss/sahi,0.538462,7,topic:deep-learning | topic:distributed | topi...
18,extism/extism,grafana/pyroscope,0.500000,5,lang:python | topic:data-analysis | topic:data...
35,JoeanAmier/XHS-Downloader,srbhr/Resume-Matcher,0.428571,3,lang:jupyter-notebook | topic:deep-learning | ...
34,grafana/pyroscope,aws-samples/bedrock-chat,0.416667,5,lang:python | topic:data-science | topic:deep-...
32,grafana/pyroscope,pulumi/pulumi-aws,0.411765,7,lang:python | topic:data-analysis | topic:data...
15,Xtremilicious/projectlearn-project-based-learning,pycaret/pycaret,0.400000,8,lang:python | topic:book | topic:computer-visi...
27,pulumi/pulumi,pycaret/pycaret,0.400000,8,lang:python | topic:data-science | topic:deep-...
20,extism/extism,aws-samples/bedrock-chat,0.400000,4,lang:python | topic:data-science | topic:machi...
39,appwrite/appwrite,srbhr/Resume-Matcher,0.400000,2,lang:jupyter-notebook | topic:machine-learning


DONE
Dashboard saved as: professional_data_mining_dashboard_DYNAMIC_COMPARE_METRICS.html
PageRank ranking file: SEPARATE_top_repositories_by_pagerank.csv
Degree Centrality ranking file: SEPARATE_top_repositories_by_degree_centrality.csv
Mapped metrics file: SEPARATE_repository_metrics_mapped_names.csv
Important: PageRank and Degree Centrality are separate. No combined final rank is used.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>